# FinAdvisor AI — Autonomous EMI & Loan Intelligence Agent
## Multi-Step Agentic Reasoning & Financial Advisory Workflow

This notebook demonstrates a **real working AI agent** implementing an explicit **multi-step Plan -> Act -> Observe -> Decide** workflow.

### Key System Features:
1. **Two Deterministic Tools**: `compare_options()` and `compute_emi(amount, rate, months)`.
2. **Stateful Conversation Memory**: Retains stated `monthly_income` and `previously_seen_options` across conversational turns.
3. **Affordability Decision Engine**: Deterministically applies the project heuristic (maximum preferred EMI $\le 30\%$ of monthly income).
4. **Transparent Execution Trace**: Step-by-step visibility into tool invocations, observations, and decisions.

In [1]:
import os
import sys
from pathlib import Path
import tempfile

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.agent import LoanAdvisorAgent
from src.memory import ConversationMemory

print("✓ Environment and modules loaded successfully.")

✓ Environment and modules loaded successfully.


---
## DEMO 1: Normal Loan Comparison & Affordability Decision

**User Goal**: *"I earn ₹50,000 per month. I need a ₹5 lakh loan. Compare a 3-year and 5-year option."*

**Agent Workflow**:
1. Understands loan parameters (Amount: ₹5,00,000, Income: ₹50,000, Tenures: 3 & 5 years).
2. Stores stated monthly income in session memory.
3. Calls tool `compare_options()` to standardize candidate loan option profiles.
4. Calls tool `compute_emi()` for each candidate option using reducing-balance formula.
5. Evaluates affordability under the 30% heuristic ($₹50,000 \times 30\% = ₹15,000$ cap).
6. Recommends Option B (5-year option at ₹10,623.52/mo) because Option A (3-year at ₹16,133.60/mo) exceeds the preferred cap.

In [1]:
demo1_memory_file = Path(tempfile.gettempdir()) / "ca1_demo1_memory.json"
demo1_memory_file.unlink(missing_ok=True)

agent1 = LoanAdvisorAgent(memory_path=demo1_memory_file, use_llm_if_available=False)

user_query_1 = "I earn ₹50,000 per month. I need a ₹5 lakh loan. Compare a 3-year and 5-year option."
result_1 = agent1.run(user_query_1)

print("=" * 70)
print("FINAL AGENT RESPONSE:")
print("=" * 70)
print(result_1["answer"])
print("\n" + "=" * 70)
print("MULTI-STEP EXECUTION TRACE:")
print("=" * 70)
print(result_1["trace"])

FINAL AGENT RESPONSE:
**Recommendation: Option B (5 Years)**

- **Monthly EMI**: ₹10,623.52 for 60 months (Rate: 10.0%)
- **Total Interest**: ₹137,411.34 (Total repayment: ₹637,411.34)
- **Affordability**: **Affordable** (EMI is 21.25% of your ₹50,000.00 monthly income, within the 30% cap of ₹15,000.00)
- **Comparison**: Option A (3 Years): EMI ₹16,133.59/mo (Interest: ₹80,809.37) | Option B (5 Years): EMI ₹10,623.52/mo (Interest: ₹137,411.34)

> *Based on the inputs you provided and the project's affordability heuristic (max 30% of income). This is an educational project heuristic, not official bank eligibility or professional financial advice.*

MULTI-STEP EXECUTION TRACE:
[USER]
I earn ₹50,000 per month. I need a ₹5 lakh loan. Compare a 3-year and 5-year option.

[AGENT PLAN]
Analyzing natural language request and extracting financial parameters.

[OBSERVE]
Extracted initial parameters.
  • income: 50000.0
  • obligations: None
  • amount: 500000.0
  • tenures_years: [3, 5]
  • rate

---
## DEMO 2: Multi-Turn Conversation & Session Memory Retention

Demonstrates state retention across three sequential conversational turns in the same session without re-asking for previously stated information.

- **Turn 1**: User provides monthly income (₹60,000).
- **Turn 2**: User requests comparison for a ₹6 Lakh loan across 3-year and 5-year tenures. Agent recalls income from memory.
- **Turn 3**: User asks a follow-up: *"What if I choose the longer tenure?"*. Agent recalls both income and previously evaluated options.

In [1]:
demo2_memory_file = Path(tempfile.gettempdir()) / "ca1_demo2_memory.json"
demo2_memory_file.unlink(missing_ok=True)
agent2 = LoanAdvisorAgent(memory_path=demo2_memory_file, use_llm_if_available=False)

# ------------------------------------------------------------
# Turn 1: State monthly income
# ------------------------------------------------------------
print("▶ TURN 1:")
t1_res = agent2.run("My monthly income is ₹60,000.")
print("Agent:", t1_res["answer"])
print("Memory Snapshot:", agent2.memory.to_dict())

# ------------------------------------------------------------
# Turn 2: Request loan comparison (income should be retrieved from memory)
# ------------------------------------------------------------
print("\n" + "-" * 50)
print("▶ TURN 2:")
t2_res = agent2.run("Compare a ₹6 lakh loan for 3 years and 5 years at 10%.")
print("Agent:", t2_res["answer"])

# ------------------------------------------------------------
# Turn 3: Follow-up question referencing earlier context
# ------------------------------------------------------------
print("\n" + "-" * 50)
print("▶ TURN 3:")
t3_res = agent2.run("What if I choose the longer tenure?")
print("Agent:", t3_res["answer"])

print("\n" + "=" * 70)
print("TURN 3 EXECUTION TRACE (Showing Memory Recall):")
print("=" * 70)
print(t3_res["trace"])

▶ TURN 1:
Agent: I noted your monthly income of ₹60,000.00. Please specify the loan amount and tenures you want to compare.
Memory Snapshot: {'monthly_income': 60000.0, 'previously_seen_options': []}

--------------------------------------------------
▶ TURN 2:
Agent: **Recommendation: Option B (5 Years)**

- **Monthly EMI**: ₹12,748.23 for 60 months (Rate: 10.0%)
- **Total Interest**: ₹164,893.61 (Total repayment: ₹764,893.61)
- **Affordability**: **Affordable** (EMI is 21.25% of your ₹60,000.00 monthly income, within the 30% cap of ₹18,000.00)
- **Comparison**: Option A (3 Years): EMI ₹19,360.31/mo (Interest: ₹96,971.24) | Option B (5 Years): EMI ₹12,748.23/mo (Interest: ₹164,893.61)

> *Based on the inputs you provided and the project's affordability heuristic (max 30% of income). This is an educational project heuristic, not official bank eligibility or professional financial advice.*

--------------------------------------------------
▶ TURN 3:
Agent: **Recommendation: Option B (5

---
## DEMO 3: Honest Failures & Edge Cases

Demonstrates genuine edge case handling:
1. **Case A (Unaffordable Loan)**: Income is ₹25,000 but requested loan is ₹10 Lakh for 2 years (EMI ₹46,144 vs 30% cap ₹7,500). Agent flags affordability violation and explains the budget gap.
2. **Case B (Missing Income)**: Stated loan options without any income in memory. Agent calculates EMIs but honestly refuses to make an affordability recommendation until income is provided.

In [1]:
# Case A: Unaffordable Loan Request
print("▶ CASE A: Unaffordable Loan Request (₹10 Lakh for 2 years on ₹25,000 income)")
case_a_mem = Path(tempfile.gettempdir()) / "ca1_case_a_memory.json"
case_a_mem.unlink(missing_ok=True)
agent_case_a = LoanAdvisorAgent(memory_path=case_a_mem, use_llm_if_available=False)
res_case_a = agent_case_a.run("My income is ₹25,000 and I want a ₹10 lakh loan for 2 years.")
print(res_case_a["answer"])

print("\n" + "-" * 50)
# Case B: Missing Income on Fresh Session
print("▶ CASE B: Missing Income on Fresh Session")
case_b_mem = Path(tempfile.gettempdir()) / "ca1_case_b_memory.json"
case_b_mem.unlink(missing_ok=True)
agent_case_b = LoanAdvisorAgent(memory_path=case_b_mem, use_llm_if_available=False)
res_case_b = agent_case_b.run("Compare a ₹5 lakh loan for 3 years and 5 years at 10%.")
print(res_case_b["answer"])
print("\nTrace excerpt:")
print(res_case_b["trace"])

▶ CASE A: Unaffordable Loan Request (₹10 Lakh for 2 years on ₹25,000 income)
**Alert: No affordable option within preferred threshold**

For a monthly income of ₹25,000.00, your maximum preferred 30% EMI budget is **₹7,500.00/month**.
The closest option is **Option A (2 Years)** at **₹46,144.93/month**, which exceeds your preferred budget by ₹38,644.93/month.

- **All Options Evaluated**: Option A (2 Years): EMI ₹46,144.93/mo (Interest: ₹107,478.23)
- **Suggestion**: Consider increasing the loan tenure or lowering the principal loan amount.

> *Based on the inputs you provided and the project's affordability heuristic (max 30% of income). This is an educational project heuristic, not official bank eligibility or professional financial advice.*

--------------------------------------------------
▶ CASE B: Missing Income on Fresh Session
I calculated the EMIs for your loan (Option A (3 Years): ₹16,133.59/mo, Option B (5 Years): ₹10,623.52/mo). However, I cannot provide an affordability r